## Recreate Jonas Plot 

In [1]:
import pandas as pd
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [ ]:
clusters_df_path = 'xx' # Some csv???
anomaly_file_path = 'xx' # Anomaly data file path
profile_times_path = 'xx' # Some csv???
output_path = '/home/onennecke/Code/Figures' 
vmin=-2.5
vmax=2.5

In [ ]:
# ---------------------------
# Top Row: Anomalies per Cluster
# ---------------------------
# Read clusters and prepare time string
clusters_df = pd.read_csv(clusters_df_path)
clusters_df["Date"] = pd.to_datetime(clusters_df["Date"])
clusters_df["Date"] = clusters_df["Date"].dt.strftime("%Y-%m-%dT12:00:00")
    
# Start counting Clusters at 1
# Load anomaly dataset and convert valid_time to string
ds = xr.open_dataset(anomaly_file_path)
ds["valid_time"] = ds["valid_time"].dt.strftime("%Y-%m-%dT%H:%M:%S")

# Merge K-means results with the anomaly dataset based on valid_time/Date
merged_data = pd.DataFrame({"valid_time": ds["valid_time"].values}).merge(
    clusters_df,
    how="left",
    left_on="valid_time",
    right_on="Date",
)

# Assign clusters as a new coordinate in the dataset
ds = ds.assign_coords(
    cluster=("valid_time", merged_data["Cluster"].fillna(-1).values)
)

# Determine valid clusters (assumed to be 5 clusters)
valid_clusters = np.sort(merged_data["Cluster"].dropna().unique())
cluster_counts = merged_data["Cluster"].value_counts()

# Compute the data extent (longitude and latitude bounds) from the dataset
lon_min = float(ds.longitude.min())
lon_max = float(ds.longitude.max())
lat_min = float(ds.latitude.min())
lat_max = float(ds.latitude.max())
data_extent = [lon_min, lon_max, lat_min, lat_max]

In [ ]:
# ---------------------------
# Bottom Row: Cluster Occurrence Plots
# ---------------------------
# For the occurrence plots, read the clusters file again and set datetime index.
clusters_occ = pd.read_csv(clusters_df_path)
clusters_occ["Date"] = pd.to_datetime(clusters_occ["Date"])
clusters_occ.set_index("Date", inplace=True)
clusters_occ["DayOfYear"] = clusters_occ.index.dayofyear

# Count occurrences per DayOfYear and cluster
daily_occurrences = (
    clusters_occ.groupby([clusters_occ["DayOfYear"], "Cluster"])
    .size()
    .unstack(fill_value=0)
)
day_of_year_counts = clusters_occ.groupby("DayOfYear").size()
normalized_daily_occurrences = daily_occurrences.div(day_of_year_counts, axis=0)

# Annual occurrences per cluster
clusters_occ["Year"] = clusters_occ.index.year
annual_occurrences = (
    clusters_occ.groupby(["Year", "Cluster"]).size().unstack(fill_value=0)
)

# Load profile time data for the single stacked bar plot
profile_times_df = pd.read_csv(profile_times_path)
profile_times_df["Time"] = pd.to_datetime(profile_times_df["Time"])
profile_times_df["Date"] = profile_times_df["Time"].dt.date
unique_dates = profile_times_df["Date"].unique()
unique_dates_df = pd.DataFrame(unique_dates, columns=["Date"])

# Merge clusters with profile times (only on field days)
clusters_occ.index = pd.to_datetime(clusters_occ.index).date
merged_occ_df = pd.merge(
    clusters_occ, unique_dates_df, left_index=True, right_on="Date", how="inner"
)
occurrence_cluster_counts = merged_occ_df["Cluster"].value_counts()
total_count = occurrence_cluster_counts.sum()
cluster_ratios = occurrence_cluster_counts / total_count
cluster_ratios_df = pd.DataFrame(cluster_ratios).T
sorted_columns = sorted(cluster_ratios_df.columns)
cluster_ratios_df = cluster_ratios_df.reindex(columns=sorted_columns)

# Define a consistent color palette for the clusters (assumes clusters are numeric)
unique_clusters = sorted(clusters_occ["Cluster"].unique())
palette = sns.color_palette(n_colors=len(unique_clusters))
cluster_palette = {cluster: color for cluster, color in zip(unique_clusters, palette)}

In [ ]:
# ---------------------------
# Create the Combined Figure with Two Rows
# ---------------------------
# Create a figure and a GridSpec with 2 rows.
fig = plt.figure(figsize=(10, 6))
outer_gs = fig.add_gridspec(2, 1, height_ratios=[1, 1], hspace=0.2)

# --- Top row: 5 columns for anomalies per cluster ---
gs_top = outer_gs[0].subgridspec(1, 5, wspace=0.2)

# Loop over each valid cluster to create an anomalies map
im = None  # To capture the last image for the shared colorbar
for i, cluster_id in enumerate(valid_clusters):
    # Use PlateCarree for the map projection
    ax = fig.add_subplot(gs_top[0, i], projection=ccrs.PlateCarree())
    
    # Restrict the map to your data region and set a white background
    ax.set_extent(data_extent, crs=ccrs.PlateCarree())
    ax.set_facecolor("white")
    
    # Calculate the mean anomalies for the given cluster
    t_anomaly = (
        ds["t_anomaly"].where(ds["cluster"] == cluster_id).mean(dim="valid_time")
    )
    z_anomaly = (
        ds["z_anomaly"].where(ds["cluster"] == cluster_id).mean(dim="valid_time")
    )
    
    # Plot temperature anomaly with PlateCarree transform (data are in lat/lon)
    im = t_anomaly.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap="coolwarm",
        vmin=vmin,
        vmax=vmax,
        add_colorbar=False,
    )
    ax.set_aspect("auto")
    
    # Add geographic features only within the data extent
    ax.coastlines()
    # ax.add_feature(cfeature.BORDERS)
    ax.add_feature(cfeature.LAND, edgecolor="black")
    
    # Add gridlines; disable labels on inner panels for clarity
    gl = ax.gridlines(draw_labels=True, linewidth=0.5, color="gray", alpha=0.6, linestyle="--")
    gl.top_labels = False
    gl.buttom_labels = True
    gl.right_labels = False
    if i != 0:
        gl.left_labels = False
        
    
    # Contour geopotential height anomaly
    z_min = np.floor(z_anomaly.min().values / 5) * 5
    z_max = np.ceil(z_anomaly.max().values / 5) * 5 + 5
    contour_levels = np.arange(z_min, z_max, 5)
    contours = ax.contour(
        z_anomaly.longitude,
        z_anomaly.latitude,
        z_anomaly.values,
        levels=contour_levels,
        colors="black",
        linewidths=1,
        transform=ccrs.PlateCarree(),
        # only solid lines
        linestyles="solid",
    )
    ax.clabel(contours, inline=True, fontsize=9, fmt="%1.0f", zorder = 2)
    

    

    count = cluster_counts.get(cluster_id, 0)
    # Annotation with both the count and GpH anomaly line
    annotation = f"n = {count}"
    ax.text(
        0.98, 0.985,  # top-right corner
        annotation,
        transform=ax.transAxes,
        fontsize=11,
        color="black",
        ha="right",
        va="top",
        bbox=dict(facecolor="white", edgecolor="black", alpha=1, boxstyle="round,pad=0.2"),
    )
    # Define a dictionary to map cluster IDs to their descriptions
    cluster_descriptions = {
        1: 'low pressure',
        2: 'zonal',
        3: 'high pressure',
        4: 'strong zonal',
        5: 'strong high pressure'
    }
    description = cluster_descriptions.get(cluster_id, '')

    ax.set_title(f"CL{int(cluster_id)}\n'{description}'", fontsize=12, fontweight="bold",  color=cluster_palette.get(cluster_id, "black"))
    
    # Add bold 'a' to the upper-left of the first-row subplots
    if i == 0:  # Assuming 5 subplots per row
        ax.text(
            -0.12, 1.1,  # Slightly outside the top-left corner
            "(a)",
            transform=ax.transAxes,
            fontsize=14,
            fontweight="bold",
            ha="left",
            va="center",
        )

# Add a shared vertical colorbar for the anomalies maps.
cbar_ax = fig.add_axes([0.91, 0.53, 0.015, 0.35])  
cbar = fig.colorbar(
    im, cax=cbar_ax, orientation="vertical", label="850 hPa Temp. Anomaly (°C)"
)
cbar.ax.tick_params(labelsize=10)
cbar.set_label("850 hPa Temp. Anomaly (°C)", fontsize=10)

# --- Bottom row: Occurrence Plots (3 panels) ---
# Reduce the horizontal spacing since we no longer need room for y-axis labels.
gs_bot = outer_gs[1].subgridspec(1, 7, wspace=0.3)  # Reduced from 0.9 to 0.1
ax_norm = fig.add_subplot(gs_bot[0, :3])
ax_ann  = fig.add_subplot(gs_bot[0, 3:6], sharey=ax_norm)  # share y-axis with ax_norm
ax_bar  = fig.add_subplot(gs_bot[0, 6:], sharey=ax_norm)    # share y-axis with ax_norm

# Add bold subplot labels "b", "c", and "d" above the three occurrence plots.
ax_norm.text(-0.032, 1.1, "(b)", transform=ax_norm.transAxes,
            fontsize=14, fontweight="bold", va="top", ha="left")
ax_ann.text(-0.03, 1.1, "(c)", transform=ax_ann.transAxes,
            fontsize=14, fontweight="bold", va="top", ha="left")
ax_bar.text(-0.035, 1.1, "(d)", transform=ax_bar.transAxes,
            fontsize=14, fontweight="bold", va="top", ha="left")

# Plot 1: Normalized Cluster Occurrences Per Day of Year
normalized_daily_occurrences.plot(
    kind="bar",
    stacked=True,
    ax=ax_norm,
    width=0.9,
    color=[cluster_palette.get(x) for x in normalized_daily_occurrences.columns],
)
ax_norm.set_ylabel("Relative Cluster Occurrence", fontsize=10)
ax_norm.set_xlabel("Day of Year (June-August)", fontsize=10)
ax_norm.set_xticks(range(0, len(normalized_daily_occurrences), 5))
ax_norm.tick_params(axis="x", labelsize=10)

# Prepare relative data for annual occurrences by dividing by 92.
annual_occurrences_relative = annual_occurrences / 92

# Average the cluster counts for first and second halve of period

first_half_abs = annual_occurrences.loc[annual_occurrences.index < 2008].mean().round(2)
second_half_abs = annual_occurrences.loc[annual_occurrences.index >= 2008].mean().round(2)

first_halve_rel = annual_occurrences_relative.loc[annual_occurrences_relative.index < 2008].mean().round(2)
second_halve_rel = annual_occurrences_relative.loc[annual_occurrences_relative.index >= 2008].mean().round(2)

# writing average clusters to dataframe
average_clusters = pd.DataFrame({
    "1991-2007 Abs": first_half_abs,
    "2008-2024 Abs": second_half_abs,
    "1991-2007 Rel": first_halve_rel,
    "2008-2024 Rel": second_halve_rel
}).T

# save average clusters to html table
average_clusters.to_html("results\\k_means\\cluster_occurence_over_time.html")


# Plot 2: Annual Cluster Occurrence (Relative)
annual_occurrences_relative.plot(
    kind="bar",
    stacked=True,
    ax=ax_ann,
    width=0.9,
    color=[cluster_palette.get(x) for x in annual_occurrences_relative.columns],
)
ax_ann.set_xticks(range(0, len(annual_occurrences_relative), 5))
ax_ann.set_xlabel("Year", fontsize=10)
ax_ann.tick_params(axis="x", labelsize=10)
# Remove the redundant y-axis label on ax_ann since y-axis is shared.
ax_ann.set_ylabel("")

# Plot 3: Single Stacked Bar Plot for Cluster Occurrence on Fielddays
cluster_order = [1, 2, 3, 5]  # use a specific stacking order
cluster_colors = [cluster_palette.get(x) for x in cluster_order]
cluster_ratios_df = cluster_ratios_df[cluster_order]  # reorder columns if needed
cluster_ratios_df.plot(
    kind="bar",
    stacked=True,
    ax=ax_bar,
    width=0.9,
    color=cluster_colors,
)
bottom_val = 0
for cluster in cluster_order:
    count = occurrence_cluster_counts.get(cluster, 0)
    ratio = count / total_count if total_count != 0 else 0
    ax_bar.text(
        0,
        bottom_val + ratio / 2,
        f"n={count}",
        ha="center",
        va="center",
        color="white",
        fontsize=9,
    )
    bottom_val += ratio
ax_bar.set_xlabel("Clusters on\nField Days", fontsize=10)
ax_bar.set_xticklabels([])

# Remove the redundant y-axis label on ax_bar since y-axis is shared.
ax_bar.set_ylabel("")

# (Optionally) adjust the y-axis limits so that all values are between 0 and 1.
ax_norm.set_ylim(0, 1)

# Create a combined legend using handles from one of the occurrence plots
handles, labels = ax_norm.get_legend_handles_labels()
labels = [f"CL{label}" for label in labels]
ax_norm.get_legend().remove()
ax_ann.get_legend().remove()
ax_bar.get_legend().remove()
fig.legend(
    handles,
    labels,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.06),
    ncol=len(labels),
    fontsize=11,
    frameon=False,
)

plt.tight_layout(rect=[0, 0.05, 0.9, 0.95])

# Ensure the output directory exists and save the figure
os.makedirs(output_path, exist_ok=True)
output_file_png = os.path.join(output_path, "combined_synoptic_and_occurrences.png")
output_file_pdf = os.path.join(output_path, "combined_synoptic_and_occurrences.pdf")
plt.savefig(output_file_png, dpi=300, bbox_inches="tight")
plt.savefig(output_file_pdf, dpi=300, bbox_inches="tight")
plt.close()
print(f"Combined plot saved to {output_file_png}")

In [ ]:
def plot_combined_synoptic_and_occurrences(
    clusters_df_path,
    anomaly_file_path,
    profile_times_path,
    output_path,
    vmin=-2.5,
    vmax=2.5,
):
    """
    Combines two figures into one canvas with two rows.

    Top row: Mean anomalies per cluster (one row, 5 columns).
    Bottom row: Three subplots for cluster occurrence metrics (normalized daily, annual, and a single stacked bar plot)
    """
    # ---------------------------
    # Top Row: Anomalies per Cluster
    # ---------------------------
    # Read clusters and prepare time string
    clusters_df = pd.read_csv(clusters_df_path)
    clusters_df["Date"] = pd.to_datetime(clusters_df["Date"])
    clusters_df["Date"] = clusters_df["Date"].dt.strftime("%Y-%m-%dT12:00:00")
    
    # Start counting Clusters at 1
    # Load anomaly dataset and convert valid_time to string
    ds = xr.open_dataset(anomaly_file_path)
    ds["valid_time"] = ds["valid_time"].dt.strftime("%Y-%m-%dT%H:%M:%S")
    
    # Merge K-means results with the anomaly dataset based on valid_time/Date
    merged_data = pd.DataFrame({"valid_time": ds["valid_time"].values}).merge(
        clusters_df,
        how="left",
        left_on="valid_time",
        right_on="Date",
    )
    
    # Assign clusters as a new coordinate in the dataset
    ds = ds.assign_coords(
        cluster=("valid_time", merged_data["Cluster"].fillna(-1).values)
    )
    
    # Determine valid clusters (assumed to be 5 clusters)
    valid_clusters = np.sort(merged_data["Cluster"].dropna().unique())
    cluster_counts = merged_data["Cluster"].value_counts()
    
    # Compute the data extent (longitude and latitude bounds) from the dataset
    lon_min = float(ds.longitude.min())
    lon_max = float(ds.longitude.max())
    lat_min = float(ds.latitude.min())
    lat_max = float(ds.latitude.max())
    data_extent = [lon_min, lon_max, lat_min, lat_max]
    
    # ---------------------------
    # Bottom Row: Cluster Occurrence Plots
    # ---------------------------
    # For the occurrence plots, read the clusters file again and set datetime index.
    clusters_occ = pd.read_csv(clusters_df_path)
    clusters_occ["Date"] = pd.to_datetime(clusters_occ["Date"])
    clusters_occ.set_index("Date", inplace=True)
    clusters_occ["DayOfYear"] = clusters_occ.index.dayofyear

    # Count occurrences per DayOfYear and cluster
    daily_occurrences = (
        clusters_occ.groupby([clusters_occ["DayOfYear"], "Cluster"])
        .size()
        .unstack(fill_value=0)
    )
    day_of_year_counts = clusters_occ.groupby("DayOfYear").size()
    normalized_daily_occurrences = daily_occurrences.div(day_of_year_counts, axis=0)
    
    # Annual occurrences per cluster
    clusters_occ["Year"] = clusters_occ.index.year
    annual_occurrences = (
        clusters_occ.groupby(["Year", "Cluster"]).size().unstack(fill_value=0)
    )
    
    # Load profile time data for the single stacked bar plot
    profile_times_df = pd.read_csv(profile_times_path)
    profile_times_df["Time"] = pd.to_datetime(profile_times_df["Time"])
    profile_times_df["Date"] = profile_times_df["Time"].dt.date
    unique_dates = profile_times_df["Date"].unique()
    unique_dates_df = pd.DataFrame(unique_dates, columns=["Date"])
    
    # Merge clusters with profile times (only on field days)
    clusters_occ.index = pd.to_datetime(clusters_occ.index).date
    merged_occ_df = pd.merge(
        clusters_occ, unique_dates_df, left_index=True, right_on="Date", how="inner"
    )
    occurrence_cluster_counts = merged_occ_df["Cluster"].value_counts()
    total_count = occurrence_cluster_counts.sum()
    cluster_ratios = occurrence_cluster_counts / total_count
    cluster_ratios_df = pd.DataFrame(cluster_ratios).T
    sorted_columns = sorted(cluster_ratios_df.columns)
    cluster_ratios_df = cluster_ratios_df.reindex(columns=sorted_columns)
    
    # Define a consistent color palette for the clusters (assumes clusters are numeric)
    unique_clusters = sorted(clusters_occ["Cluster"].unique())
    palette = sns.color_palette(n_colors=len(unique_clusters))
    cluster_palette = {cluster: color for cluster, color in zip(unique_clusters, palette)}
    
    # ---------------------------
    # Create the Combined Figure with Two Rows
    # ---------------------------
    # Create a figure and a GridSpec with 2 rows.
    fig = plt.figure(figsize=(10, 6))
    outer_gs = fig.add_gridspec(2, 1, height_ratios=[1, 1], hspace=0.2)
    
    # --- Top row: 5 columns for anomalies per cluster ---
    gs_top = outer_gs[0].subgridspec(1, 5, wspace=0.2)
    
    # Loop over each valid cluster to create an anomalies map
    im = None  # To capture the last image for the shared colorbar
    for i, cluster_id in enumerate(valid_clusters):
        # Use PlateCarree for the map projection
        ax = fig.add_subplot(gs_top[0, i], projection=ccrs.PlateCarree())
        
        # Restrict the map to your data region and set a white background
        ax.set_extent(data_extent, crs=ccrs.PlateCarree())
        ax.set_facecolor("white")
        
        # Calculate the mean anomalies for the given cluster
        t_anomaly = (
            ds["t_anomaly"].where(ds["cluster"] == cluster_id).mean(dim="valid_time")
        )
        z_anomaly = (
            ds["z_anomaly"].where(ds["cluster"] == cluster_id).mean(dim="valid_time")
        )
        
        # Plot temperature anomaly with PlateCarree transform (data are in lat/lon)
        im = t_anomaly.plot(
            ax=ax,
            transform=ccrs.PlateCarree(),
            cmap="coolwarm",
            vmin=vmin,
            vmax=vmax,
            add_colorbar=False,
        )
        ax.set_aspect("auto")
        
        # Add geographic features only within the data extent
        ax.coastlines()
        ax.add_feature(cfeature.BORDERS)
        ax.add_feature(cfeature.LAND, edgecolor="black")
        
        # Add gridlines; disable labels on inner panels for clarity
        gl = ax.gridlines(draw_labels=True, linewidth=0.5, color="gray", alpha=0.6, linestyle="--")
        gl.top_labels = False
        gl.buttom_labels = True
        gl.right_labels = False
        if i != 0:
            gl.left_labels = False
            
        
        # Contour geopotential height anomaly
        z_min = np.floor(z_anomaly.min().values / 5) * 5
        z_max = np.ceil(z_anomaly.max().values / 5) * 5 + 5
        contour_levels = np.arange(z_min, z_max, 5)
        contours = ax.contour(
            z_anomaly.longitude,
            z_anomaly.latitude,
            z_anomaly.values,
            levels=contour_levels,
            colors="black",
            linewidths=1,
            transform=ccrs.PlateCarree(),
            # only solid lines
            linestyles="solid",
        )
        ax.clabel(contours, inline=True, fontsize=9, fmt="%1.0f", zorder = 2)
        

        

        count = cluster_counts.get(cluster_id, 0)
        # Annotation with both the count and GpH anomaly line
        annotation = f"n = {count}"
        ax.text(
            0.98, 0.985,  # top-right corner
            annotation,
            transform=ax.transAxes,
            fontsize=11,
            color="black",
            ha="right",
            va="top",
            bbox=dict(facecolor="white", edgecolor="black", alpha=1, boxstyle="round,pad=0.2"),
        )
        # Define a dictionary to map cluster IDs to their descriptions
        cluster_descriptions = {
            1: 'low pressure',
            2: 'zonal',
            3: 'high pressure',
            4: 'strong zonal',
            5: 'strong high pressure'
        }
        description = cluster_descriptions.get(cluster_id, '')

        ax.set_title(f"CL{int(cluster_id)}\n'{description}'", fontsize=12, fontweight="bold",  color=cluster_palette.get(cluster_id, "black"))
        
        # Add bold 'a' to the upper-left of the first-row subplots
        if i == 0:  # Assuming 5 subplots per row
            ax.text(
                -0.12, 1.1,  # Slightly outside the top-left corner
                "(a)",
                transform=ax.transAxes,
                fontsize=14,
                fontweight="bold",
                ha="left",
                va="center",
            )

    # Add a shared vertical colorbar for the anomalies maps.
    cbar_ax = fig.add_axes([0.91, 0.53, 0.015, 0.35])  
    cbar = fig.colorbar(
        im, cax=cbar_ax, orientation="vertical", label="850 hPa Temp. Anomaly (°C)"
    )
    cbar.ax.tick_params(labelsize=10)
    cbar.set_label("850 hPa Temp. Anomaly (°C)", fontsize=10)
    
    # --- Bottom row: Occurrence Plots (3 panels) ---
    # Reduce the horizontal spacing since we no longer need room for y-axis labels.
    gs_bot = outer_gs[1].subgridspec(1, 7, wspace=0.3)  # Reduced from 0.9 to 0.1
    ax_norm = fig.add_subplot(gs_bot[0, :3])
    ax_ann  = fig.add_subplot(gs_bot[0, 3:6], sharey=ax_norm)  # share y-axis with ax_norm
    ax_bar  = fig.add_subplot(gs_bot[0, 6:], sharey=ax_norm)    # share y-axis with ax_norm

    # Add bold subplot labels "b", "c", and "d" above the three occurrence plots.
    ax_norm.text(-0.032, 1.1, "(b)", transform=ax_norm.transAxes,
                fontsize=14, fontweight="bold", va="top", ha="left")
    ax_ann.text(-0.03, 1.1, "(c)", transform=ax_ann.transAxes,
                fontsize=14, fontweight="bold", va="top", ha="left")
    ax_bar.text(-0.035, 1.1, "(d)", transform=ax_bar.transAxes,
                fontsize=14, fontweight="bold", va="top", ha="left")
    
    # Plot 1: Normalized Cluster Occurrences Per Day of Year
    normalized_daily_occurrences.plot(
        kind="bar",
        stacked=True,
        ax=ax_norm,
        width=0.9,
        color=[cluster_palette.get(x) for x in normalized_daily_occurrences.columns],
    )
    ax_norm.set_ylabel("Relative Cluster Occurrence", fontsize=10)
    ax_norm.set_xlabel("Day of Year (June-August)", fontsize=10)
    ax_norm.set_xticks(range(0, len(normalized_daily_occurrences), 5))
    ax_norm.tick_params(axis="x", labelsize=10)

    # Prepare relative data for annual occurrences by dividing by 92.
    annual_occurrences_relative = annual_occurrences / 92

    # Average the cluster counts for first and second halve of period
    
    first_half_abs = annual_occurrences.loc[annual_occurrences.index < 2008].mean().round(2)
    second_half_abs = annual_occurrences.loc[annual_occurrences.index >= 2008].mean().round(2)

    first_halve_rel = annual_occurrences_relative.loc[annual_occurrences_relative.index < 2008].mean().round(2)
    second_halve_rel = annual_occurrences_relative.loc[annual_occurrences_relative.index >= 2008].mean().round(2)

    # writing average clusters to dataframe
    average_clusters = pd.DataFrame({
        "1991-2007 Abs": first_half_abs,
        "2008-2024 Abs": second_half_abs,
        "1991-2007 Rel": first_halve_rel,
        "2008-2024 Rel": second_halve_rel
    }).T
    
    # save average clusters to html table
    average_clusters.to_html("results\\k_means\\cluster_occurence_over_time.html")


    # Plot 2: Annual Cluster Occurrence (Relative)
    annual_occurrences_relative.plot(
        kind="bar",
        stacked=True,
        ax=ax_ann,
        width=0.9,
        color=[cluster_palette.get(x) for x in annual_occurrences_relative.columns],
    )
    ax_ann.set_xticks(range(0, len(annual_occurrences_relative), 5))
    ax_ann.set_xlabel("Year", fontsize=10)
    ax_ann.tick_params(axis="x", labelsize=10)
    # Remove the redundant y-axis label on ax_ann since y-axis is shared.
    ax_ann.set_ylabel("")

    # Plot 3: Single Stacked Bar Plot for Cluster Occurrence on Fielddays
    cluster_order = [1, 2, 3, 5]  # use a specific stacking order
    cluster_colors = [cluster_palette.get(x) for x in cluster_order]
    cluster_ratios_df = cluster_ratios_df[cluster_order]  # reorder columns if needed
    cluster_ratios_df.plot(
        kind="bar",
        stacked=True,
        ax=ax_bar,
        width=0.9,
        color=cluster_colors,
    )
    bottom_val = 0
    for cluster in cluster_order:
        count = occurrence_cluster_counts.get(cluster, 0)
        ratio = count / total_count if total_count != 0 else 0
        ax_bar.text(
            0,
            bottom_val + ratio / 2,
            f"n={count}",
            ha="center",
            va="center",
            color="white",
            fontsize=9,
        )
        bottom_val += ratio
    ax_bar.set_xlabel("Clusters on\nField Days", fontsize=10)
    ax_bar.set_xticklabels([])
    
    # Remove the redundant y-axis label on ax_bar since y-axis is shared.
    ax_bar.set_ylabel("")

    # (Optionally) adjust the y-axis limits so that all values are between 0 and 1.
    ax_norm.set_ylim(0, 1)
    
    # Create a combined legend using handles from one of the occurrence plots
    handles, labels = ax_norm.get_legend_handles_labels()
    labels = [f"CL{label}" for label in labels]
    ax_norm.get_legend().remove()
    ax_ann.get_legend().remove()
    ax_bar.get_legend().remove()
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.06),
        ncol=len(labels),
        fontsize=11,
        frameon=False,
    )
    
    plt.tight_layout(rect=[0, 0.05, 0.9, 0.95])
    
    # Ensure the output directory exists and save the figure
    os.makedirs(output_path, exist_ok=True)
    output_file_png = os.path.join(output_path, "combined_synoptic_and_occurrences.png")
    output_file_pdf = os.path.join(output_path, "combined_synoptic_and_occurrences.pdf")
    plt.savefig(output_file_png, dpi=300, bbox_inches="tight")
    plt.savefig(output_file_pdf, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Combined plot saved to {output_file_png}")